In [18]:
import numpy as np

k, q = 1.38e-23, 1.60e-19
Isc, Voc, Ns = 8, 38, 54            
Ki, Kv, n = 0.0032, -0.123, 1.3            
Rsh, Rs, Ion = 415.405, 0.221, 9.825e-8    
T = 25 + 273.15
Vt = (Ns * k * T) / q 

def solve_voltage(I, G):
    I_pv = (Isc + Ki * (T - 298.15)) * (G / 1000)
    I_o_adj = Ion * ((T / 298.15)**3) * np.exp(((q * 1.12) / (n * k)) * (1/298.15 - 1/T))
    
    if I > I_pv + I_o_adj: 
        return 0.0
    
    v_guess = 0.0
    for _ in range(5): 
        arg = (I_pv + I_o_adj - I - v_guess / Rsh) / I_o_adj
        if arg <= 0:
            v_guess = 0
            break
        v_guess = n * Vt * np.log(arg) - I * Rs
    return max(0.0, v_guess)



def obj_4S(I, g1, g2, g3, g4):
    V = (solve_voltage(I, g1) + solve_voltage(I, g2) + solve_voltage(I, g3) + solve_voltage(I, g4))
    return V * I

def obj_4P(V, g1, g2, g3, g4):
    i_mesh = np.linspace(0, Isc, 350)
    v1 = [solve_voltage(i, g1) for i in i_mesh]
    v2 = [solve_voltage(i, g2) for i in i_mesh]
    v3 = [solve_voltage(i, g3) for i in i_mesh]
    v4 = [solve_voltage(i, g4) for i in i_mesh]
    
    i1 = np.interp(V, v1[::-1], i_mesh[::-1], left=Isc, right=0)
    i2 = np.interp(V, v2[::-1], i_mesh[::-1], left=Isc, right=0)
    i3 = np.interp(V, v3[::-1], i_mesh[::-1], left=Isc, right=0)
    i4 = np.interp(V, v4[::-1], i_mesh[::-1], left=Isc, right=0)
    return V * (i1 + i2 + i3 + i4)

def obj_2S2P(V, g1, g2, g3, g4):
    i_mesh = np.linspace(0, Isc, 350)
    v_strA = [solve_voltage(i, g1) + solve_voltage(i, g2) for i in i_mesh]
    v_strB = [solve_voltage(i, g3) + solve_voltage(i, g4) for i in i_mesh]
    
    iA = np.interp(V, v_strA[::-1], i_mesh[::-1], left=Isc, right=0)
    iB = np.interp(V, v_strB[::-1], i_mesh[::-1], left=Isc, right=0)
    return V * (iA + iB)

def obj_2P2S(I, g1, g2, g3, g4):
    i_mesh = np.linspace(0, Isc, 350)
    v1 = [solve_voltage(i, g1) for i in i_mesh]
    v2 = [solve_voltage(i, g2) for i in i_mesh]  
    v3 = [solve_voltage(i, g3) for i in i_mesh]
    v4 = [solve_voltage(i, g4) for i in i_mesh]
    
    v_shared = np.linspace(0, Voc, 350)
    i1 = np.interp(v_shared, v1[::-1], i_mesh[::-1], left=Isc, right=0)
    i2 = np.interp(v_shared, v2[::-1], i_mesh[::-1], left=Isc, right=0)
    i3 = np.interp(v_shared, v3[::-1], i_mesh[::-1], left=Isc, right=0)
    i4 = np.interp(v_shared, v4[::-1], i_mesh[::-1], left=Isc, right=0)
    
    V_blockA = np.interp(I, (i1 + i2)[::-1], v_shared[::-1], left=0, right=0)
    V_blockB = np.interp(I, (i3 + i4)[::-1], v_shared[::-1], left=0, right=0)
    return (V_blockA + V_blockB) * I


def run_custom_pso(target_function, lb, ub, num_particles=100, max_iter=30):
    w = 0.5   
    c1 = 2    
    c2 = 2  
    
    X = np.random.uniform(lb, ub, num_particles)
    Vel = np.zeros(num_particles)
    
    pbest_X = np.copy(X)
    pbest_Fit = np.array([target_function(pos) for pos in X])
    
    gbest_idx = np.argmax(pbest_Fit)
    gbest_X = pbest_X[gbest_idx]
    gbest_Fit = pbest_Fit[gbest_idx]
    


    for _ in range(max_iter):
        for i in range(num_particles):
            r1 = np.random.rand()
            r2 = np.random.rand()
            
            Vel[i] = (w * Vel[i] +  c1 * r1 * (pbest_X[i] - X[i]) + c2 * r2 * (gbest_X - X[i]))
            
            X[i] = X[i] + Vel[i]
            
            X[i] = np.clip(X[i], lb, ub)
            current_fitness = target_function(X[i])
            
            if current_fitness > pbest_Fit[i]:
                pbest_Fit[i] = current_fitness
                pbest_X[i] = X[i]
                
                if current_fitness > gbest_Fit:
                    gbest_Fit = current_fitness
                    gbest_X = X[i]
                    
    return gbest_Fit


def find_overall_mpp_with_pso(g1, g2, g3, g4):
    print(f"\nInsolation Levels: G1={g1}, G2={g2}, G3={g3}, G4={g4}")

    p_4s = run_custom_pso(lambda x: obj_4S(x, g1, g2, g3, g4), lb=0, ub=Isc)

    p_4p = run_custom_pso(lambda x: obj_4P(x, g1, g2, g3, g4), lb=0, ub=Voc)

    p_2s2p = run_custom_pso(lambda x: obj_2S2P(x, g1, g2, g3, g4), lb=0, ub=2*Voc)

    p_2p2s = run_custom_pso(lambda x: obj_2P2S(x, g1, g2, g3, g4), lb=0, ub=2*Isc)

    combinations = ['4S', '4P', '2S2P', '2P2S']
    powers = [p_4s, p_4p, p_2s2p, p_2p2s]
    
    for comb, p in zip(combinations, powers):
        print(f"Max Power for {comb}: {p:.2f} W")
        
    best_idx = np.argmax(powers)
    print(f" GLOBAL MPP: {combinations[best_idx]} Topology with {powers[best_idx]:.2f} W")


find_overall_mpp_with_pso(g1=100, g2=100, g3=100, g4=200)


Insolation Levels: G1=100, G2=100, G3=100, G4=200
Max Power for 4S: 69.12 W
Max Power for 4P: 84.16 W
Max Power for 2S2P: 68.24 W
Max Power for 2P2S: 71.50 W
 GLOBAL MPP: 4P Topology with 84.16 W
